In [62]:
pip install bertopic

In [63]:
from datasets import load_dataset

dataset = load_dataset(
"MaartenGr/arxiv_nlp", split="train"
)
abstracts = dataset["Abstracts"]

Convertite testo -> vettori densi

In [64]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("thenlper/gte-small")
embeddings = model.encode(
abstracts,
show_progress_bar=True
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/1405 [00:00<?, ?it/s]

In [65]:
import time

# Misurazione del tempo per l'embedding
start_time_embedding = time.time()

In [66]:
end_time_embedding = time.time()
duration_embedding = end_time_embedding - start_time_embedding
print(f"Tempo di Embedding: {duration_embedding:.2f} secondi")

# Misurazione del tempo per la riduzione UMAP
start_time_umap = time.time()

Tempo di Embedding: 0.01 secondi


Ridurre dimensionalità mantenendo struttura

#### `n_components`

Specifies the number of dimensions in the reduced embedding space.

* Original sentence embeddings generated by `gte-small` have 384 dimensions.
* UMAP compresses them into a lower-dimensional representation.

Reducing dimensionality decreases computational cost and helps HDBSCAN identify clusters more effectively.

---

#### `min_dist`

Controls how closely points can be packed together in the low-dimensional space.

* Lower values allow points to form very dense clusters.
* Higher values spread points more evenly across the space.

---

#### `metric`

Defines the distance metric used to measure similarity between documents during dimensionality reduction.

* Cosine distance compares the angle between embedding vectors rather than their magnitude.
* Euclidean defines the distance metric based on the straight-line distance between two points in a vector space.

In practice:

```text
Small cosine distance  → documents are semantically similar
Large cosine distance  → documents are semantically different
Small Euclidean distance → points are close together
Large Euclidean distance → points are far apart
```

---

### `n_neighbors`
Determines how many neighboring points UMAP considers when learning the structure of the data.

* Small values (e.g. 5–10)
  * → focus on very local structure
  * → more detailed, fine-grained clusters
  * → can produce more fragmented groupings

* Large values (e.g. 30–100)
  * → preserve more global structure
  * → smoother, more general clusters
  * → different groups may merge
---

#### `random_state`

Sets the random seed used by UMAP.

* Ensures reproducible results across multiple runs.
---

### Output

```python
reduced = umap_model.fit_transform(embeddings)
```

The method:

1. Learns a low-dimensional representation of the embedding space.
2. Preserves the local neighborhood structure of the original embeddings.
3. Returns a matrix of shape:

```text
(number_of_documents, 5)
```

where each row corresponds to a document represented in the reduced 5-dimensional space.


In [67]:
from umap import UMAP

umap_model = UMAP(
n_components=5,
min_dist=0.0,
metric="cosine",
random_state=42
)
reduced = umap_model.fit_transform(embeddings)
reduced

array([[ 0.7700457 , -0.06036054,  6.76378   ,  8.712606  ,  5.436405  ],
       [ 0.77679116, -0.05894969,  6.7535443 ,  8.68613   ,  5.417076  ],
       [ 0.47709882, -0.29589164,  6.6893616 ,  8.341756  ,  5.1701283 ],
       ...,
       [ 0.8219463 , -1.1528326 ,  6.4293013 ,  8.811807  ,  2.1996872 ],
       [ 0.7756178 , -1.066488  ,  6.2299886 ,  8.074769  ,  3.8570921 ],
       [ 0.8212472 , -1.1471066 ,  6.430105  ,  8.816107  ,  2.203316  ]],
      dtype=float32)

In [68]:
end_time_umap = time.time()
duration_umap = end_time_umap - start_time_umap
print(f"Tempo di Riduzione UMAP: {duration_umap:.2f} secondi")

# Misurazione del tempo per il clustering HDBSCAN
start_time_hdbscan = time.time()

Tempo di Riduzione UMAP: 53.68 secondi


#### `min_cluster_size=50`

Defines the minimum number of points required to form a cluster.

* Clusters smaller than 50 documents are considered noise or outliers.
* Larger values produce fewer, more general topics.
* Smaller values produce more, finer-grained topics.


```text
Small min_cluster_size → many small topics
Large min_cluster_size → few large topics
```

---

#### `metric="euclidean"`

Specifies the distance metric used to compute distances between points.

* Euclidean distance measures the straight-line distance between points in space.
* It works well after UMAP reduces embeddings into a low-dimensional geometric space.

```text
Small Euclidean distance → points are close → likely same cluster
Large Euclidean distance → points are far → likely different clusters
```

---

#### `cluster_selection_method="eom" | "leaf"`

Controls how clusters are selected from the HDBSCAN cluster hierarchy.

* **`"eom"` (Excess of Mass)**

  * Default and most stable option
  * Produces a smaller number of broader, more robust clusters
  * Prefers stability over granularity

* **`"leaf"`**

  * Produces more detailed clusters
  * Selects the most fine-grained structure in the hierarchy
  * Can result in many small topics

---

#### `min_samples`

Controls how conservative the clustering is in labeling points as noise.

* Higher values → more points labeled as outliers (-1)
* Lower values → more points assigned to clusters

If `None`:

* Defaults to `min_samples = min_cluster_size`
* Leads to more conservative clustering

---

### Output

```python
clusters = hdbscan_model.fit_predict(reduced)
```

The result is an array of cluster labels:

```text
[0, 0, 2, 1, -1, 2, ...]
```

* Each number represents a cluster (topic)
* `-1` represents noise or outliers (documents that do not clearly belong to any cluster)


In [69]:
from hdbscan import HDBSCAN
import numpy as np

hdbscan_model = HDBSCAN(
min_cluster_size=50,
metric="euclidean",
cluster_selection_method="eom",
min_samples= None,
)
clusters = hdbscan_model.fit_predict(reduced)
print("Outliers =", np.sum(clusters == -1) / len(clusters))

print(f"Number of distinct topics found: {len(set(clusters) - {-1})}")
clusters

Outliers = 0.3456806603039
Number of distinct topics found: 158


array([ -1,  -1,  -1, ...,  -1, 146,  -1])

In [70]:
end_time_hdbscan = time.time()
duration_hdbscan = end_time_hdbscan - start_time_hdbscan
print(f"Tempo di Clustering HDBSCAN: {duration_hdbscan:.2f} secondi")

total_duration = duration_embedding + duration_umap + duration_hdbscan
print(f"\nTempo Totale Computazionale (Embedding + UMAP + HDBSCAN): {total_duration:.2f} secondi")

Tempo di Clustering HDBSCAN: 2.67 secondi

Tempo Totale Computazionale (Embedding + UMAP + HDBSCAN): 56.36 secondi


Estrarre topic interpretabili

In [71]:
from bertopic import BERTopic

topic_model = BERTopic(
embedding_model=model,
umap_model=umap_model,
hdbscan_model=hdbscan_model)

topics, probs = topic_model.fit_transform(
abstracts, embeddings
)

### Tempo Computazionale: Embedding, Riduzione e Clustering

In [72]:
topic_model.visualize_barchart(
    top_n_topics=10
)

In [ ]:
top_5_topics = topic_model.get_topic_info()[topic_model.get_topic_info()["Topic"] != -1].head(5)["Topic"].tolist()

for topic_id in top_5_topics:
    keywords = [word for word, _ in topic_model.get_topic(topic_id)[:5]]
    print("="*80)
    print(f"ANALISI QUALITATIVA: TOPIC {topic_id}")
    print(f"Parole chiave c-TF-IDF: {', '.join(keywords)}")
    print("="*80)

    rep_docs = topic_model.get_representative_docs(topic_id)

    for i, doc in enumerate(rep_docs[:3]):
        print(f"📄 Documento Rappresentativo #{i+1}:")
        print(f"{doc[:400]}...")
        print("-" * 50)
    print("\n" * 2)

🔬 ANALISI QUALITATIVA: TOPIC 0
Parole chiave c-TF-IDF: speech, asr, recognition, end, acoustic
📄 Documento Rappresentativo #1:
  Recent advances in text-to-speech (TTS) led to the development of flexible
multi-speaker end-to-end TTS systems. We extend state-of-the-art
attention-based automatic speech recognition (ASR) systems with synthetic audio
generated by a TTS system trained only on the ASR corpora itself. ASR and TTS
systems are built separately to show that text-only data can be used to enhance
existing end-to-end ...
--------------------------------------------------
📄 Documento Rappresentativo #2:
  Conversational context information, higher-level knowledge that spans across
sentences, can help to recognize a long conversation. However, existing speech
recognition models are typically built at a sentence level, and thus it may not
capture important conversational context information. The recent progress in
end-to-end speech recognition enables integrating context with other av